# FiCO Model Fine-Tuning (Colab / NVIDIA GPU)
Bu defter, FiCO modelini **Unsloth** kütüphanesi kullanarak hızlı ve verimli bir şekilde eğitmek için tasarlanmıştır.

### 1. Kurulum
Aşağıdaki hücreyi çalıştırarak gerekli kütüphaneleri yükleyin.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 2. Model ve Tokenizer Yükleme
Llama-3.2-3B modelini 4-bit kuantizasyon ile yüklüyoruz.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-3b-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Veri Setini Hazırlama
Mac'inizde ürettiğiniz `fico_train_dataset.jsonl` dosyasını sol taraftaki dosya menüsünden Colab'a yükleyin. Ardından bu hücreyi çalıştırın.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": "fico_train_dataset.jsonl"}, split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Alpaca formatına benzer bir yapı kullanıyoruz
        text = f"### Soru:\n{instruction}\n\n### Bağlam:\n{input}\n\n### Cevap:\n{output}" + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

### 4. Eğitim (Training)
Eğitimi başlatıyoruz. Bu işlem genellikle 5-10 dakika sürer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Küçük veri seti için yeterlidir
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 5. Modeli Kaydetme
Eğitilen adapter dosyalarını bilgisayarınıza indirmek için hazırlıyoruz.

In [ ]:
model.save_pretrained_lora("lora_model")
tokenizer.save_pretrained("lora_model")

# Dosyaları sıkıştırıp indirilebilir hale getirelim
!zip -r lora_model.zip lora_model